# Dubois surface inversion evaluation on UAVSAR C3 data

In this notebook we run the Dubois surface inversion on the UAVSAR C3 product using the provided incidence-angle stream. The radar frequency is derived from the NetCDF metadata.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
from pathlib import Path

import numpy as np
import xarray as xr
from dask.diagnostics import ProgressBar

from polsarpro.io import open_netcdf_beam
from polsarpro.physical_inversion import dubois_surface_inversion

# change to your local C-PolSARpro install dir
# Kept for consistency with the other dev notebooks, but unused in this
# UAVSAR-only variant.
c_psp_dir = "/home/c_psp/Soft/bin/"
os.environ["PATH"] += os.pathsep + f"{c_psp_dir}/data_process_sngl/"
os.environ["PATH"] += os.pathsep + f"{c_psp_dir}/data_convert/"

input_uavsar_nc = Path("/data/psp/test_files/blackw_20801_22010_006_220507_L090_CX_02_ML5X5.nc")
incidence_angle_file = Path(
    "/data/psp/test_files/blackw_20801_22010_006_220507_L090_CX_02.inc"
)
file_out = Path("/data/psp/res/test_dubois_surface_inversion_uavsar.nc")

with xr.open_dataset(input_uavsar_nc) as raw_ds:
    center_wavelength_cm = raw_ds["metadata"].attrs["Annotation:CenterWavelength"]

# Dubois expects GHz; UAVSAR metadata stores the center wavelength in cm.
freq_ghz = 29.9792458 / float(center_wavelength_cm)

# Data-driven starting point from the multilooked UAVSAR scene:
# - hv/vv 10th percentile ≈ -12.0 dB
# - hh/vv 25th percentile ≈ -4.5 dB
# This keeps only the lower tails of the cross-pol and co-pol ratios,
# which is a good first pass for suppressing volume and double-bounce
# contributions.
thresh1_db = -12.0
thresh2_db = -4.5
calibration_coeff = 1.0

print(f"Derived radar frequency: {freq_ghz:.6f} GHz")


## Load UAVSAR data and incidence angle raster

In [ ]:
C3 = open_netcdf_beam(input_uavsar_nc)
row_dim, col_dim = C3.dims
# nlines = C3.sizes[row_dim]
# ncols = C3.sizes[col_dim]

# use lines from .ann file
# inc.set_rows                                      (pixels)    = 12344                  ; INC Lines
# inc.set_cols                                      (pixels)    = 11248                  ; INC Samples

nlines = 12344
ncols = 11248

# The incidence-angle file is a little-endian float32 stream in radians.
# Read only the raster-sized prefix and reshape it to the UAVSAR grid.
incidence_angle_values = np.fromfile(
    incidence_angle_file,
    dtype="<f4",
    count=nlines * ncols,
)

# reduce to the same dimensions as multilooked 5x5 image
incidence_angle_values[incidence_angle_values==-10000] = np.nan
incidence_angle = xr.DataArray(
    incidence_angle_values.reshape(nlines, ncols),
    dims=(row_dim, col_dim),
    name="incidence_angle",
).coarsen(y=5, x=5, boundary="trim").mean()

## Apply the surface inversion

In [ ]:
if file_out.exists():
    file_out.unlink()

with ProgressBar():
    out_py = dubois_surface_inversion(
        C3,
        incidence_angle=incidence_angle,
        freq_ghz=freq_ghz,
        thresh1=thresh1_db,
        thresh2=thresh2_db,
        calibration_coeff=calibration_coeff,
    )
    out_py.to_netcdf(file_out)


## Inspect results

In [ ]:
out_py = xr.open_dataset(file_out)
out_py

## Plot masks and outputs

In [ ]:
import matplotlib.pyplot as plt

mask_vars = [
    ("dubois_mask_in", "Mask in"),
    ("dubois_mask_out", "Mask out"),
    ("dubois_mask_valid_in_out", "Valid in/out mask"),
]
result_vars = [
    ("dubois_ks", "Ks"),
    ("dubois_er", "Er"),
    ("dubois_mv", "Mv"),
]

fig, axes = plt.subplots(2, 3, figsize=(18, 10), constrained_layout=True)
for ax, (var, title) in zip(axes[0], mask_vars):
    im = out_py[var].plot.imshow(ax=ax, cmap="magma", add_colorbar=False)
    ax.set_title(title)
    ax.set_xlabel("")
    ax.set_ylabel("")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

for ax, (var, title) in zip(axes[1], result_vars):
    im = out_py[var].plot.imshow(ax=ax, cmap="viridis", robust=True, add_colorbar=False)
    ax.set_title(title)
    ax.set_xlabel("")
    ax.set_ylabel("")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.show()


In [ ]:
from polsarpro.util import pauli_rgb

pauli_rgb(C3).plot.imshow(origin="upper")